# DeepLOB: Deep Convolutional Neural Networks for Limit Order Books

### Authors: Zihao Zhang, Stefan Zohren and Stephen Roberts
Oxford-Man Institute of Quantitative Finance, Department of Engineering Science, University of Oxford

This notebook reproduces the DeepLOB paper [2] on the FI-2010 dataset [1], with additional statistical analyses:
- Training and evaluation for all 5 prediction horizons (k=1,2,3,4,5)
- Confusion matrices
- t-tests and statistical significance tests on trading signals
- Cohen's Kappa, Matthews Correlation Coefficient
- Cross-horizon performance comparison

### References:
[1] Ntakaris et al. Benchmark dataset for mid-price forecasting of LOB. Journal of Forecasting, 2018.

[2] Zhang Z, Zohren S, Roberts S. DeepLOB. IEEE Transactions on Signal Processing. 2019.

## 1. Setup & Imports

In [ ]:
# Ensure ocean user packages are on path inside kernel
import sys, os
PYUSER = "/ocean/projects/mth250011p/xxiao7/pyuser"
PYVER = f"{sys.version_info.major}.{sys.version_info.minor}"
site_pkg = os.path.join(PYUSER, "lib", f"python{PYVER}", "site-packages")
if site_pkg not in sys.path:
    sys.path.insert(0, site_pkg)

# Keep matplotlib non-interactive for batch execution
import matplotlib
matplotlib.use("Agg")
os.makedirs("/ocean/projects/mth250011p/xxiao7/mpl_cache", exist_ok=True)
os.environ["MPLCONFIGDIR"] = "/ocean/projects/mth250011p/xxiao7/mpl_cache"


In [ ]:
import os
import sys
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from datetime import datetime
from tqdm import tqdm
from scipy import stats

from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    cohen_kappa_score, matthews_corrcoef,
    precision_recall_fscore_support
)

import torch
import torch.nn.functional as F
from torch.utils import data
from torchinfo import summary
import torch.nn as nn
import torch.optim as optim

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Paths
BASE_DIR   = '/ocean/projects/mth250011p/xxiao7/DeepLOB'
DATA_DIR   = os.path.join(BASE_DIR, 'data')
MODEL_DIR  = os.path.join(BASE_DIR, 'models')
RESULT_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Data Preparation

FI-2010 dataset: no-auction, decimal-precision normalised.
- Rows 0–39: 10 bid/ask price+volume levels (features)
- Rows 40–44: labels at horizons k=1,2,3,4,5 (10,20,30,50,100 events)
- Labels: 1=down, 2=stationary, 3=up → mapped to 0,1,2

In [ ]:
dec_data  = np.loadtxt(os.path.join(DATA_DIR, 'Train_Dst_NoAuction_DecPre_CF_7.txt'))
dec_train = dec_data[:, :int(np.floor(dec_data.shape[1] * 0.8))]
dec_val   = dec_data[:, int(np.floor(dec_data.shape[1] * 0.8)):]

dec_test1 = np.loadtxt(os.path.join(DATA_DIR, 'Test_Dst_NoAuction_DecPre_CF_7.txt'))
dec_test2 = np.loadtxt(os.path.join(DATA_DIR, 'Test_Dst_NoAuction_DecPre_CF_8.txt'))
dec_test3 = np.loadtxt(os.path.join(DATA_DIR, 'Test_Dst_NoAuction_DecPre_CF_9.txt'))
dec_test  = np.hstack((dec_test1, dec_test2, dec_test3))

print(f'Train: {dec_train.shape}, Val: {dec_val.shape}, Test: {dec_test.shape}')

# Label distribution for each horizon
horizon_names = {0: 'k=1 (10 events)', 1: 'k=2 (20 events)',
                 2: 'k=3 (30 events)', 3: 'k=4 (50 events)', 4: 'k=5 (100 events)'}
print('\nLabel distribution in test set (per horizon):')
for k_idx in range(5):
    labels = dec_test[-5+k_idx, :] - 1  # 0,1,2
    unique, counts = np.unique(labels, return_counts=True)
    dist = dict(zip(['down','stationary','up'], counts))
    print(f'  {horizon_names[k_idx]}: {dist}')

In [ ]:
def prepare_x(data):
    return np.array(data[:40, :].T)

def get_label(data):
    return np.array(data[-5:, :].T)

def data_classification(X, Y, T):
    N, D = X.shape
    dataY = Y[T - 1:N]
    dataX = np.zeros((N - T + 1, T, D))
    for i in range(T, N + 1):
        dataX[i - T] = X[i - T:i, :]
    return dataX, dataY


class LOBDataset(data.Dataset):
    """FI-2010 LOB dataset for a given prediction horizon k."""
    def __init__(self, raw_data, k, num_classes=3, T=100):
        self.k = k
        self.num_classes = num_classes
        self.T = T

        x = prepare_x(raw_data).astype(np.float32)
        y = get_label(raw_data)
        x, y = data_classification(x, y, T)
        y = y[:, k] - 1            # 0=down, 1=stat, 2=up
        self.length = len(x)

        x = torch.from_numpy(x)
        self.x = torch.unsqueeze(x, 1)  # (N, 1, T, D)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


# Quick sanity check for k=4 (horizon k=5, 100 events)
ds_tmp = LOBDataset(dec_train, k=4)
print(f'Train dataset (k=4): x={ds_tmp.x.shape}, y={ds_tmp.y.shape}')

## 3. Model Architecture (DeepLOB)

In [ ]:
class DeepLOB(nn.Module):
    """DeepLOB: CNN + Inception + LSTM for LOB mid-price prediction."""
    def __init__(self, y_len=3):
        super().__init__()
        self.y_len = y_len

        # Convolution block 1 (feature extraction over price levels)
        self.conv1 = nn.Sequential(
            nn.Conv2d(1,  32, kernel_size=(1,2), stride=(1,2)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )
        # Convolution block 2
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1,2), stride=(1,2)),
            nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.Tanh(), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.Tanh(), nn.BatchNorm2d(32),
        )
        # Convolution block 3 (combine across levels)
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1,10)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4,1)),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(32),
        )

        # Inception module (multi-scale temporal features)
        self.inp1 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=(3,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=(5,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )
        self.inp3 = nn.Sequential(
            nn.MaxPool2d((3,1), stride=(1,1), padding=(1,0)),
            nn.Conv2d(32, 64, kernel_size=(1,1), padding='same'),
            nn.LeakyReLU(0.01), nn.BatchNorm2d(64),
        )

        # LSTM for temporal modelling
        self.lstm = nn.LSTM(input_size=192, hidden_size=64, num_layers=1, batch_first=True)
        self.fc1  = nn.Linear(64, y_len)

    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), 64).to(x.device)
        c0 = torch.zeros(1, x.size(0), 64).to(x.device)

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)

        x = torch.cat([self.inp1(x), self.inp2(x), self.inp3(x)], dim=1)
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(-1, x.shape[1], x.shape[2])

        x, _ = self.lstm(x, (h0, c0))
        x = x[:, -1, :]
        return torch.softmax(self.fc1(x), dim=1)


# Print architecture summary for one model
test_model = DeepLOB(y_len=3).to(device)
summary(test_model, (1, 1, 100, 40))
del test_model

## 4. Training Function

In [ ]:
def batch_gd(model, criterion, optimizer, train_loader, val_loader, epochs, model_path):
    """Training loop with validation monitoring and best-model checkpointing."""
    train_losses = np.zeros(epochs)
    val_losses   = np.zeros(epochs)
    best_val_loss  = np.inf
    best_val_epoch = 0

    for ep in tqdm(range(epochs), desc='Epochs'):
        model.train()
        t0 = datetime.now()
        train_loss = []
        for inputs, targets in train_loader:
            inputs  = inputs.to(device, dtype=torch.float)
            targets = targets.to(device, dtype=torch.int64)
            optimizer.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimizer.step()
            train_loss.append(loss.item())
        train_loss = np.mean(train_loss)

        model.eval()
        val_loss = []
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs  = inputs.to(device, dtype=torch.float)
                targets = targets.to(device, dtype=torch.int64)
                loss = criterion(model(inputs), targets)
                val_loss.append(loss.item())
        val_loss = np.mean(val_loss)

        train_losses[ep] = train_loss
        val_losses[ep]   = val_loss

        if val_loss < best_val_loss:
            torch.save(model, model_path)
            best_val_loss  = val_loss
            best_val_epoch = ep
            print(f'  [Epoch {ep+1}] Model saved (val_loss={val_loss:.4f})')

        dt = datetime.now() - t0
        print(f'Epoch {ep+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | '
              f'Best ep: {best_val_epoch+1} | Time: {dt}')

    return train_losses, val_losses

## 5. Train & Evaluate for All Prediction Horizons

We train one DeepLOB model per horizon k ∈ {1,2,3,4,5}.

In [ ]:
EPOCHS     = 50
BATCH_SIZE = 64
LR         = 1e-4
NUM_CLASSES = 3
T           = 100   # lookback window

# k indices: 0→10ev, 1→20ev, 2→30ev, 3→50ev, 4→100ev
K_VALUES = [0, 1, 2, 3, 4]
K_LABELS = ['k=1 (10ev)', 'k=2 (20ev)', 'k=3 (30ev)', 'k=4 (50ev)', 'k=5 (100ev)']

all_results = {}   # store metrics per k

In [ ]:
import gc
for k_idx, k_label in zip(K_VALUES, K_LABELS):
    print(f'\n{"="*60}')
    print(f'Training for horizon {k_label}  (k_idx={k_idx})')
    print(f'{"-"*60}')

    # Skip training if predictions already saved
    pred_file = os.path.join(RESULT_DIR, f"preds_k{k_idx}.npz")
    model_path = os.path.join(MODEL_DIR, f"deeplob_k{k_idx}.pt")
    if os.path.exists(pred_file) and os.path.exists(model_path):
        print(f"  -> Already done, loading from disk.")
        pdata = np.load(pred_file)
        y_true, y_pred, y_probs = pdata["y_true"], pdata["y_pred"], pdata["y_probs"]
        acc   = accuracy_score(y_true, y_pred)
        kappa = cohen_kappa_score(y_true, y_pred)
        mcc   = matthews_corrcoef(y_true, y_pred)
        prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, average=None, labels=[0,1,2])
        prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted")
        train_losses = np.load(os.path.join(RESULT_DIR, f"losses_k{k_idx}.npz"))["train"] if os.path.exists(os.path.join(RESULT_DIR, f"losses_k{k_idx}.npz")) else np.zeros(50)
        val_losses   = np.load(os.path.join(RESULT_DIR, f"losses_k{k_idx}.npz"))["val"]   if os.path.exists(os.path.join(RESULT_DIR, f"losses_k{k_idx}.npz")) else np.zeros(50)
        all_results[k_idx] = {"label": k_label, "train_losses": train_losses, "val_losses": val_losses,
            "accuracy": acc, "kappa": kappa, "mcc": mcc,
            "precision": prec, "recall": rec, "f1": f1, "support": sup,
            "precision_w": prec_w, "recall_w": rec_w, "f1_w": f1_w}
        print(classification_report(y_true, y_pred, target_names=["Down","Stationary","Up"], digits=4))
        # Plot confusion matrix
        cm_val = confusion_matrix(y_true, y_pred)
        cm_norm = cm_val.astype(float) / cm_val.sum(axis=1, keepdims=True)
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        ConfusionMatrixDisplay(cm_val,  display_labels=["Down","Stationary","Up"]).plot(ax=axes[0], colorbar=False)
        ConfusionMatrixDisplay(cm_norm, display_labels=["Down","Stationary","Up"]).plot(ax=axes[1], colorbar=False)
        axes[0].set_title(f"Confusion Matrix (counts) — {k_label}")
        axes[1].set_title(f"Confusion Matrix (row-normalised) — {k_label}")
        plt.tight_layout(); fig.savefig(os.path.join(RESULT_DIR, f"cm_k{k_idx}.png"), dpi=120); plt.show()
        torch.cuda.empty_cache(); gc.collect()
        continue

    # ---- Datasets ----
    ds_train = LOBDataset(dec_train, k=k_idx, num_classes=NUM_CLASSES, T=T)
    ds_val   = LOBDataset(dec_val,   k=k_idx, num_classes=NUM_CLASSES, T=T)
    ds_test  = LOBDataset(dec_test,  k=k_idx, num_classes=NUM_CLASSES, T=T)

    train_loader = data.DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
    val_loader   = data.DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
    test_loader  = data.DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    print(f'Samples — train: {len(ds_train)}, val: {len(ds_val)}, test: {len(ds_test)}')

    # ---- Model ----
    model_path = os.path.join(MODEL_DIR, f'deeplob_k{k_idx}.pt')
    model      = DeepLOB(y_len=NUM_CLASSES).to(device)
    criterion  = nn.CrossEntropyLoss()
    optimizer  = optim.Adam(model.parameters(), lr=LR)

    # ---- Train ----
    train_losses, val_losses = batch_gd(
        model, criterion, optimizer,
        train_loader, val_loader,
        epochs=EPOCHS, model_path=model_path
    )

    # ---- Plot loss curves ----
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(train_losses, label='Train loss')
    ax.plot(val_losses,   label='Val loss')
    ax.set_title(f'Loss curves — {k_label}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')
    ax.legend()
    plt.tight_layout()
    fig.savefig(os.path.join(RESULT_DIR, f'loss_k{k_idx}.png'), dpi=120)
    plt.show()

    # ---- Evaluate on test set ----
    best_model = torch.load(model_path, map_location=device)
    best_model.eval()

    all_targets, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs  = inputs.to(device, dtype=torch.float)
            targets = targets.to(device, dtype=torch.int64)
            probs   = best_model(inputs)
            _, preds = torch.max(probs, 1)
            all_targets.append(targets.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

    y_true  = np.concatenate(all_targets)
    y_pred  = np.concatenate(all_preds)
    y_probs = np.concatenate(all_probs)

    acc   = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc   = matthews_corrcoef(y_true, y_pred)
    prec, rec, f1, sup = precision_recall_fscore_support(y_true, y_pred, average=None, labels=[0,1,2])
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

    print(f'\n--- Test Results ({k_label}) ---')
    print(f'Accuracy:            {acc:.4f}')
    print(f"Cohen's Kappa:       {kappa:.4f}")
    print(f'MCC:                 {mcc:.4f}')
    print(f'Weighted F1:         {f1_w:.4f}')
    print('\nClassification report:')
    print(classification_report(y_true, y_pred,
                                target_names=['Down','Stationary','Up'], digits=4))

    # Save per-horizon results
    all_results[k_idx] = {
        'label': k_label, 'train_losses': train_losses, 'val_losses': val_losses,
        'y_true': y_true, 'y_pred': y_pred, 'y_probs': y_probs,
        'accuracy': acc, 'kappa': kappa, 'mcc': mcc,
        'precision': prec, 'recall': rec, 'f1': f1, 'support': sup,
        'precision_w': prec_w, 'recall_w': rec_w, 'f1_w': f1_w,
    }

    # ---- Confusion matrix ----
    cm_val = confusion_matrix(y_true, y_pred)
    cm_norm = cm_val.astype(float) / cm_val.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay(cm_val,  display_labels=['Down','Stationary','Up']).plot(ax=axes[0], colorbar=False)
    ConfusionMatrixDisplay(cm_norm, display_labels=['Down','Stationary','Up']).plot(ax=axes[1], colorbar=False)
    axes[0].set_title(f'Confusion Matrix (counts) — {k_label}')
    axes[1].set_title(f'Confusion Matrix (row-normalised) — {k_label}')
    plt.tight_layout()
    fig.savefig(os.path.join(RESULT_DIR, f'cm_k{k_idx}.png'), dpi=120)
    plt.show()


    # Free memory before next horizon
    del ds_train, ds_val, ds_test, train_loader, val_loader, test_loader
    del model, best_model, optimizer, criterion
    del inputs, targets, probs, preds
    # Save raw predictions to disk to avoid keeping large arrays in memory
    np.savez_compressed(os.path.join(RESULT_DIR, f'losses_k{k_idx}.npz'), train=train_losses, val=val_losses)
    np.savez_compressed(os.path.join(RESULT_DIR, f'preds_k{k_idx}.npz'),
                        y_true=y_true, y_pred=y_pred, y_probs=y_probs)
    # Keep only summary metrics in memory (not raw arrays)
    all_results[k_idx].pop('y_true', None)
    all_results[k_idx].pop('y_pred', None)
    all_results[k_idx].pop('y_probs', None)
    torch.cuda.empty_cache()
    gc.collect()


# Persist results
with open(os.path.join(RESULT_DIR, 'all_results.pkl'), 'wb') as f:
    pickle.dump(all_results, f)

print('\n✓ All horizons trained and evaluated.')

## 6. Cross-Horizon Performance Comparison

In [ ]:
# Build summary table
rows = []
for k_idx in K_VALUES:
    r = all_results[k_idx]
    rows.append({
        'Horizon': r['label'],
        'Accuracy': f"{r['accuracy']:.4f}",
        "Cohen's κ": f"{r['kappa']:.4f}",
        'MCC': f"{r['mcc']:.4f}",
        'F1 (Down)': f"{r['f1'][0]:.4f}",
        'F1 (Stat.)': f"{r['f1'][1]:.4f}",
        'F1 (Up)': f"{r['f1'][2]:.4f}",
        'Weighted F1': f"{r['f1_w']:.4f}",
    })

df_summary = pd.DataFrame(rows).set_index('Horizon')
print('Performance comparison across prediction horizons:')
display(df_summary)
df_summary.to_csv(os.path.join(RESULT_DIR, 'performance_summary.csv'))

In [ ]:
# Bar-chart comparison
metrics = ['accuracy', 'kappa', 'mcc', 'f1_w']
metric_labels = ['Accuracy', "Cohen's κ", 'MCC', 'Weighted F1']
x = np.arange(len(K_VALUES))
width = 0.2

fig, ax = plt.subplots(figsize=(13, 5))
for i, (m, ml) in enumerate(zip(metrics, metric_labels)):
    vals = [all_results[k][m] for k in K_VALUES]
    ax.bar(x + i * width, vals, width, label=ml)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(K_LABELS, rotation=15)
ax.set_ylabel('Score')
ax.set_title('DeepLOB: Performance vs. Prediction Horizon')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, 'comparison_horizons.png'), dpi=120)
plt.show()

## 7. Statistical Tests on Trading Signals

We treat the model's directional predictions as trading signals and apply:
1. **One-sample t-test** — test whether each signal class (up/down/stationary) is predicted at rates significantly different from random (1/3).
2. **Two-sample t-test (Welch's)** — test whether mean predicted probability for the true class significantly exceeds chance.
3. **Precision t-test** — test whether per-class precision is significantly > 1/3 (naive baseline).
4. **Kolmogorov–Smirnov test** — test whether predicted probabilities for true-positive vs. true-negative samples come from different distributions.
5. **Cohen's h** — effect size for proportion tests.

In [ ]:
CLASS_NAMES = ['Down', 'Stationary', 'Up']
RANDOM_RATE = 1.0 / 3.0

def cohens_h(p1, p2):
    """Effect size for difference between two proportions."""
    return 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))


stat_rows = []

for k_idx in K_VALUES:
    r = all_results[k_idx]
    preds_f = np.load(os.path.join(RESULT_DIR, f'preds_k{k_idx}.npz'))
    y_true  = preds_f['y_true']
    y_pred  = preds_f['y_pred']
    y_probs = preds_f['y_probs']
    n       = len(y_true)

    print(f"\n{'='*65}")
    print(f"Statistical Tests — {r['label']}")
    print(f"{'='*65}")

    # ── Test 1: One-sample t-test on predicted class probability ──
    # H0: mean predicted prob for each class == 1/3 (chance)
    print("\n[T1] One-sample t-test: mean predicted prob vs. chance (1/3)")
    print(f"  {'Class':<15} {'Mean Prob':>10} {'t-stat':>10} {'p-value':>12} {'Sig':>6}")
    for c, cname in enumerate(CLASS_NAMES):
        probs_c = y_probs[:, c]
        t, p = stats.ttest_1samp(probs_c, RANDOM_RATE)
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f"  {cname:<15} {probs_c.mean():>10.4f} {t:>10.3f} {p:>12.4e} {sig:>6}")
        stat_rows.append({'Horizon': r['label'], 'Test': 'T1-mean-prob', 'Class': cname,
                          'Statistic': t, 'p_value': p, 'Significant': p < 0.05})

    # ── Test 2: Two-sample Welch t-test ──
    # H0: predicted prob for true class same between correct and incorrect predictions
    print("\n[T2] Welch t-test: prob(true class) for correct vs. incorrect predictions")
    print(f"  {'Class':<15} {'μ correct':>10} {'μ incorrect':>12} {'t-stat':>10} {'p-value':>12} {'Sig':>6}")
    for c, cname in enumerate(CLASS_NAMES):
        mask_true  = (y_true == c)
        if mask_true.sum() < 10:
            continue
        prob_true_cls = y_probs[mask_true, c]
        correct   = prob_true_cls[y_pred[mask_true] == c]
        incorrect = prob_true_cls[y_pred[mask_true] != c]
        if len(correct) < 5 or len(incorrect) < 5:
            continue
        t, p = stats.ttest_ind(correct, incorrect, equal_var=False)
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f"  {cname:<15} {correct.mean():>10.4f} {incorrect.mean():>12.4f} {t:>10.3f} {p:>12.4e} {sig:>6}")
        stat_rows.append({'Horizon': r['label'], 'Test': 'T2-welch', 'Class': cname,
                          'Statistic': t, 'p_value': p, 'Significant': p < 0.05})

    # ── Test 3: Precision z-test (proportion test) ──
    # H0: precision == 1/3 (random classifier)
    print("\n[T3] Proportion z-test: precision vs. random baseline (1/3)")
    print(f"  {'Class':<15} {'Precision':>10} {'z-stat':>10} {'p-value':>12} {'Cohen h':>10} {'Sig':>6}")
    for c, cname in enumerate(CLASS_NAMES):
        pred_c = (y_pred == c)
        n_pred = pred_c.sum()
        if n_pred < 10:
            continue
        prec_c = (y_true[pred_c] == c).mean()
        # z-test for proportion
        from statsmodels.stats.proportion import proportions_ztest as _pzt
        z_stat, p_val = _pzt(count=int(prec_c * n_pred),
                                                 nobs=n_pred,
                                                 value=RANDOM_RATE,
                                                 alternative='larger')
        h = cohens_h(prec_c, RANDOM_RATE)
        sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
        print(f"  {cname:<15} {prec_c:>10.4f} {z_stat:>10.3f} {p_val:>12.4e} {h:>10.4f} {sig:>6}")
        stat_rows.append({'Horizon': r['label'], 'Test': 'T3-precision-z', 'Class': cname,
                          'Statistic': z_stat, 'p_value': p_val, 'Significant': p_val < 0.05})

    # ── Test 4: KS test ──
    # H0: distribution of model confidence is the same for correct vs. incorrect predictions
    print("\n[T4] Kolmogorov–Smirnov test: confidence distribution, correct vs. incorrect")
    max_conf = y_probs.max(axis=1)
    correct_mask   = (y_true == y_pred)
    incorrect_mask = ~correct_mask
    ks_stat, ks_p = stats.ks_2samp(max_conf[correct_mask], max_conf[incorrect_mask])
    sig = '***' if ks_p < 0.001 else ('**' if ks_p < 0.01 else ('*' if ks_p < 0.05 else 'ns'))
    print(f"  KS statistic: {ks_stat:.4f}, p-value: {ks_p:.4e}  {sig}")
    print(f"  Mean confidence — Correct: {max_conf[correct_mask].mean():.4f}, "
          f"Incorrect: {max_conf[incorrect_mask].mean():.4f}")
    stat_rows.append({'Horizon': r['label'], 'Test': 'T4-KS', 'Class': 'all',
                      'Statistic': ks_stat, 'p_value': ks_p, 'Significant': ks_p < 0.05})

df_stats = pd.DataFrame(stat_rows)
df_stats.to_csv(os.path.join(RESULT_DIR, 'statistical_tests.csv'), index=False)
print('\n✓ Statistical tests complete. Results saved.')

## 8. Confidence Distribution Visualisation

In [ ]:
fig, axes = plt.subplots(1, len(K_VALUES), figsize=(18, 4), sharey=True)

for ax, k_idx in zip(axes, K_VALUES):
    r = all_results[k_idx]
    preds_f = np.load(os.path.join(RESULT_DIR, f'preds_k{k_idx}.npz'))
    y_true  = preds_f['y_true']
    y_pred  = preds_f['y_pred']
    y_probs = preds_f['y_probs']

    correct   = y_probs.max(1)[y_true == y_pred]
    incorrect = y_probs.max(1)[y_true != y_pred]

    ax.hist(correct,   bins=40, alpha=0.6, label='Correct',   density=True)
    ax.hist(incorrect, bins=40, alpha=0.6, label='Incorrect', density=True)
    ax.set_title(K_LABELS[k_idx], fontsize=9)
    ax.set_xlabel('Max probability')
    if k_idx == K_VALUES[0]:
        ax.set_ylabel('Density')
        ax.legend(fontsize=8)

fig.suptitle('Model Confidence: Correct vs. Incorrect Predictions', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, 'confidence_dist.png'), dpi=120)
plt.show()

## 9. Confusion Matrices (All Horizons — Summary)

In [ ]:
fig, axes = plt.subplots(2, len(K_VALUES), figsize=(18, 9))

for col, k_idx in enumerate(K_VALUES):
    r = all_results[k_idx]
    _p = np.load(os.path.join(RESULT_DIR, f'preds_k{k_idx}.npz'))
    y_true = _p['y_true']
    y_pred = _p['y_pred']
    cm_raw  = confusion_matrix(y_true, y_pred)
    cm_norm = cm_raw.astype(float) / cm_raw.sum(axis=1, keepdims=True)

    for row, (cm_data, fmt) in enumerate([(cm_raw, 'd'), (cm_norm, '.2f')]):
        ax = axes[row, col]
        sns.heatmap(cm_data, annot=True, fmt=fmt, ax=ax,
                    xticklabels=['D','S','U'], yticklabels=['D','S','U'],
                    cmap='Blues', cbar=False)
        ax.set_title(K_LABELS[k_idx], fontsize=8)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True' if col == 0 else '')

axes[0, 0].set_ylabel('Counts\nTrue')
axes[1, 0].set_ylabel('Row-normalised\nTrue')
fig.suptitle('Confusion Matrices — All Prediction Horizons', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, 'all_cm_summary.png'), dpi=120)
plt.show()

## 10. Final Summary

In [ ]:
print('\n' + '='*65)
print('  DeepLOB Reproduction — Final Performance Summary')
print('='*65)
display(df_summary)

print(f'\nResults and models saved to:')
print(f'  Models:  {MODEL_DIR}')
print(f'  Results: {RESULT_DIR}')

# Show best horizon
best_k = max(K_VALUES, key=lambda k: all_results[k]['accuracy'])
print(f"\nBest accuracy: {all_results[best_k]['label']} — {all_results[best_k]['accuracy']:.4f}")
best_k_f1 = max(K_VALUES, key=lambda k: all_results[k]['f1_w'])
print(f"Best weighted F1: {all_results[best_k_f1]['label']} — {all_results[best_k_f1]['f1_w']:.4f}")